In [68]:
# %pip install pdf2image
%pip install --upgrade langchain-core langchain-openai


  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 0.3.35
    Uninstalling langchain-openai-0.3.35:
      Successfully uninstalled langchain-openai-0.3.35

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [84]:
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel,Field
from langgraph.graph import START,END,StateGraph
from typing import Literal,Any
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
import json
from types import SimpleNamespace
from io import BytesIO
from pdf2image import convert_from_path
llm = ChatOpenAI(model="gpt-4.1")
import openai
import base64
from pathlib import Path



def get_pdf_page_as_base64_image(pdf_path: str, page_num: int = 0) -> str:
    # Convert first page of PDF to a PIL Image object
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
    if not images:
        raise ValueError("Could not extract pages from PDF.")
    
    # Save image to an in-memory buffer as PNG
    buffered = BytesIO()
    images[0].save(buffered, format="PNG")
    
    # Encode buffer bytes to base64 string
    return base64.b64encode(buffered.getvalue()).decode("utf-8")



class GeneralState(BaseModel):
    imporvement_type:str = ""
    claim_data: Any = None 
    document_base64: str = ""
    final_decision:str = ""
    reasons_for_failure:list[str] = Field(default_factory=list)


improvement_data = {
    4: {"reward_amount":2000, "imporvement_type":"Air Source Heat Pump"},
    5: {"reward_amount":2000, "imporvement_type":"Ground Source Heat Pump"},
    6: {"reward_amount":500, "imporvement_type":"Biomass Pellet Boiler"},
    7: {"reward_amount":1000, "imporvement_type":"Solar Panels"},
    8: {"reward_amount":1000, "imporvement_type":"Battery for solar storage"},
    9: {"reward_amount":500, "imporvement_type":"Biomass pellet boiler"},
    10: {"reward_amount":500, "imporvement_type":"A rated glazing(where replacing single glazing)"},
    11: {"reward_amount":500, "imporvement_type":"Secondary glazing"},
    12: {"reward_amount":500, "imporvement_type":"Cavity wall insulation"},
    22: {"reward_amount":500, "imporvement_type":"Flat roof insulation"},
}


def getBase64ListByCaseRef(caseId:str) -> str:
    pdf_file = "../data/docs/solar_invoice_diff_name.pdf"
    # actual api need to be called for pega 
    base64_pdf = get_pdf_page_as_base64_image(pdf_file)
    return base64_pdf
    

def fetch_and_set_claim_details_node(state:GeneralState) -> GeneralState:
    claimData = state.claim_data
    improvementData = improvement_data[claimData.PromotionOfImprovementId]
    userName = claimData.claimRaisedBy.UserPayload.name
    homeAddress = claimData.groupServices.serviceDetails.propertyAddress
    state.claim_data = {
        "improvement_data":improvementData,
        "user_name":userName,
        "home_address":homeAddress
    }
    documentBase64 = getBase64ListByCaseRef(claimData.CaseRef)
    state.document_base64 = documentBase64
    return state

def generate_verification_system_prompt(
    expected_name: str,
    expected_address: str,
    minimum_date: str,
    minimum_amount: float,
    expected_improvement_type: str,
    currency_symbol: str = "£"
) -> str:
    """
    Generates a structured system prompt for an LLM document verification agent,
    injecting dynamic constraints into the instruction template.
    """
    
    system_prompt = f"""You are an expert Document Verification Agent inside an automated processing pipeline. Your task is to analyze the attached base64 document and strictly validate its contents against the provided User Constraints.

        ### VARIABLES TO CHECK AGAINST
        Use the following context variables for verification:
        - Expected Customer Name: {expected_name}
        - Expected Property Address: {expected_address}
        - Minimum Cut-off Date: {minimum_date}
        - Minimum Invoice Amount: {currency_symbol}{minimum_amount:,.2f}
        - Expected Improvement Type: {expected_improvement_type}

        ### EVALUATION CRITERIA
        Extract the fields from the provided document and perform these validation steps:
        1. CUSTOMER NAME CHECK: Perform a case-insensitive match.
        2. PROPERTY ADDRESS CHECK: Verify that the physical delivery/installation address matches the Expected Property Address.
        3. DATE RANGE CHECK: The invoice date MUST be strictly greater than (after) the Minimum Cut-off Date.
        4. AMOUNT CHECK: The total invoice amount MUST be strictly greater than the Minimum Invoice Amount. Ensure currency symbols are normalized before comparison.
        5. IMPROVEMENT TYPE RELATION CHECK: The items or services listed in the invoice must be completely related or logically map to the Expected Improvement Type.

        ### OUTPUT STRUCTURE
        Output your final analysis in a strict, raw JSON block. Do not include markdown code ticks (```json) or conversational filler.

        {{
            "extracted_data": {{
                "invoice_name": "string",
                "invoice_address": "string",
                "invoice_date": "YYYY-MM-DD",
                "invoice_amount": float,
                "improvement_type": "string"
            }},
            "verification_results": {{
                "name_matched": boolean,
                "address_matched": boolean,
                "date_valid": boolean,
                "amount_valid": boolean,
                "improvement_type_related": boolean
            }},
            "reasons_for_failure": [],
            "final_status": "PASSED" | "FAILED"
        }}"""

    return system_prompt





def process_result_node(state: GeneralState) -> GeneralState:
    # 1. Initialize the official native OpenAI client
    # It automatically hooks to your loaded environment variables
    client = openai.OpenAI()

    user_constraints = {
        "expected_name": state.claim_data["user_name"],
        "expected_address": state.claim_data["home_address"],
        "minimum_date": "2025-12-31",
        "minimum_amount": float(state.claim_data["improvement_data"]["reward_amount"]),
        "expected_improvement_type": state.claim_data["improvement_data"]["imporvement_type"]
    }
    
    dynamic_system_prompt = generate_verification_system_prompt(**user_constraints)

    # 2. Invoke the completion API using direct OpenAI payload patterns
    response = client.chat.completions.create(
        model="gpt-4o",  # Ensured vision compatibility configuration
        messages=[
            {
                "role": "system",
                "content": dynamic_system_prompt
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", 
                        "text": "Please run the document verification workflow on this attached invoice."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{state.document_base64}"
                        }
                    }
                ]
            }
        ],
        response_format={"type": "json_object"}  # Forces strict JSON structure output matches
    )

    # 3. Extract the text block content out securely
    raw_content = response.choices[0].message.content
    
    # 4. Parse JSON metadata string structure 
    ai_response_dict = json.loads(raw_content)
    
    state.final_decision = ai_response_dict.get("final_status", "FAILED")
    state.reasons_for_failure = ai_response_dict.get("reasons_for_failure", [])

    return state


def route(state:GeneralState) -> Literal["success_node", "failed_node"]:
    if(state.final_decision == "PASSED"):
        return "success_node"
    else:
        return "failed_node"
    

def success_node(state:GeneralState) -> GeneralState:
    # update api to claim approved mimic the colleauge step
    print("claim approved")


def failed_node(state:GeneralState)-> GeneralState:
    print("failed to verify for below reasons and extracted ")
    print(state.reasons_for_failure)
    # update api with following reason  for rejection and all data extracted by AI
    return state

def send_email_about_rejection(state:GeneralState) ->GeneralState:
    print("sending email now to colleauge about what is missing in this claim")
    # need to implement sending email about human in loop needed.
    return state



graph = StateGraph(GeneralState)
graph.add_node("fetch_and_set_claim_details_node", fetch_and_set_claim_details_node)
graph.add_node("process_result_node", process_result_node)
graph.add_node("success_node", success_node)
graph.add_node("failed_node", failed_node)
graph.add_node("send_email_about_rejection", send_email_about_rejection)

graph.add_edge(START,"fetch_and_set_claim_details_node")
graph.add_edge("fetch_and_set_claim_details_node","process_result_node")
graph.add_conditional_edges("process_result_node",route)
graph.add_edge("success_node",END)
graph.add_edge("failed_node","send_email_about_rejection")
graph.add_edge("send_email_about_rejection",END)

final_graph = graph.compile()


mock_claim_payload = {
    "PromotionOfImprovementId": 7,  # Maps to 'Battery for solar storage', £1000 threshold
    "CaseRef": "CASE-98432",
    "claimRaisedBy": {
        "UserPayload": {
            "name": "John Doe"
        }
    },
    "groupServices": {
        "serviceDetails": {
            "propertyAddress": "123 Maple Avenue, Springfield, IL 62701"
        }
    }
}

def dict_to_namespace(data):
    if isinstance(data, dict):
        return SimpleNamespace(**{k: dict_to_namespace(v) for k, v in data.items()})
    elif isinstance(data, list):
        return [dict_to_namespace(i) for i in data]
    return data

claim_data_object = dict_to_namespace(mock_claim_payload)

initial_state = {
    "imporvement_type": "",
    "claim_data": claim_data_object,  # Passes object allowing dot-notation
    "document_base64": "",
    "final_decision": "",
    "reasons_for_failure": []
}

# 4. Invoke the LangGraph pipeline
print("Starting Agentic Document Workflow Execution...")
graph_config = {"configurable": {"thread_id": "verification_session_1"}}
output_state = final_graph.invoke(initial_state, config=graph_config)

print("\nWorkflow Execution Finished.")

Starting Agentic Document Workflow Execution...


APITimeoutError: Request timed out.